# 🧠 Brain Tumor Classification Training

This notebook demonstrates how to train brain tumor classification models using the refactored codebase.

## Available Models
- ResNet152V2
- DenseNet201
- VGG16
- EfficientNetV2S
- ConvNeXtBase

## Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")

In [ ]:
# Import required modules
from src.classification import ClassificationTrainer
from src.utils.config import load_config
from src.utils.constants import AVAILABLE_MODELS

print("Available models:", AVAILABLE_MODELS)

## Configuration

You can either:
1. Use the default YAML config file
2. Override specific parameters

In [ ]:
# Load default configuration
config = load_config()

# Or override specific parameters
config['model_name'] = 'ResNet152V2'  # Change to any available model
config['epochs'] = 1  # Reduce for faster training
config['batch_size'] = 32

print("Training Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")


## Initialize Trainer

In [ ]:
trainer = ClassificationTrainer(
    model_name=config['model_name'],
    batch_size=config['batch_size'],
    learning_rate=config['learning_rate'],
    config=config,
)

print(f"✅ Trainer initialized for {config['model_name']}")

## Prepare Data

In [ ]:
# Prepare training, validation, and test datasets
trainer.prepare_data(
    val_split=config.get('val_split', 0.2),
    use_augmentation=config.get('augmentation', {}).get('enabled', True),
)

print("✅ Data prepared successfully")
print(f"   Training samples: {len(trainer.train_df)}")
print(f"   Validation samples: {len(trainer.val_df)}")
print(f"   Test samples: {len(trainer.test_df)}")

## Build Model

In [ ]:
# Build the model architecture
trainer.build()

# Display model summary
trainer.model.summary()

## Compile Model

In [ ]:
# Compile with optimizer and metrics
trainer.compile()

print("✅ Model compiled successfully")

## Train Model

**Note:** Training will take time depending on your hardware.
- With GPU: ~1-3 hours for 50 epochs
- With CPU: Much longer (not recommended)

In [ ]:
# Start training
history = trainer.train(
    epochs=config['epochs'],
    use_class_weights=config.get('use_class_weights', True),
)

print("✅ Training complete!")

## Visualize Training History

In [ ]:
from src.utils.visualization import plot_training_history
import matplotlib.pyplot as plt

# Plot training curves
plot_training_history(history.history)
plt.show()

## Evaluate on Test Set

In [ ]:
# Evaluate on test dataset
test_results = trainer.evaluate()

print("\n📊 Test Results:")
for metric, value in test_results.items():
    print(f"   {metric}: {value:.4f}")

## Generate Confusion Matrix

In [ ]:
from src.classification.inference import ClassificationEvaluator

# Create evaluator
evaluator = ClassificationEvaluator(trainer.model, trainer.test_ds)

# Plot confusion matrix
evaluator.plot_confusion_matrix(normalize=True)
plt.show()

# Print classification report
evaluator.print_classification_report()

## Save Model (Optional)

Model is automatically saved during training via ModelCheckpoint callback.
You can also manually save:

In [ ]:
# Manual save (usually not needed)
trainer.save()

print(f"✅ Model saved to: {trainer.checkpoint_path}")

## Quick Prediction Test

In [ ]:
from src.classification.inference import TumorClassifier
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image as keras_image

# Load a test image
test_image_path = trainer.test_df.iloc[0]['path']  # First test image

# Create classifier
classifier = TumorClassifier(
    str(trainer.checkpoint_path),
    model_name=config['model_name']
)

# Predict
result = classifier.predict(test_image_path)

# Display
img = keras_image.load_img(test_image_path, target_size=(224, 224))
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis('off')
plt.title(f"{result['class']} ({result['confidence']:.1f}%)")
plt.show()

print(f"Predicted: {result['class']}")
print(f"Confidence: {result['confidence']:.2f}%")
print(f"True label: {trainer.test_df.iloc[0]['class']}")

## Summary

Training complete! Your model weights and logs are saved in:
- **Weights**: `weights/classification/{model_name}_best_weights.keras`
- **Logs**: `logs/classification/{model_name}_training_log.csv`

You can now:
1. Use the trained model for predictions
2. Evaluate on new data
3. Train a different architecture by changing `model_name`